# Final statistical validation: EXP-04 to EXP-06 policy delta

This notebook tests only the reconciled policy delta. It does not change the model, thresholds, dataset, economic formulas, FLEX formulation, or walk-forward procedure.

In [ ]:
import os,sys,subprocess,hashlib
from pathlib import Path
REPO=Path('/content/FICOS-Platform')
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/SSOHEB/FICOS-Platform.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pandas','numpy','scikit-learn','matplotlib','pyyaml'],check=True)
os.chdir(REPO); sys.path.insert(0,str(REPO))
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest,f_regression
DATA=REPO/'data'/'modeling_dataset.csv'; df=pd.read_csv(DATA); df['date']=pd.to_datetime(df['date']); df=df.sort_values('date').reset_index(drop=True)
print('git_commit',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()); print('dataset_sha256',hashlib.sha256(DATA.read_bytes()).hexdigest())

In [ ]:
VESSELS=['panamax','supramax','handy','cape']; FEATURES=[c for c in df.columns if c!='date' and not c.startswith('target_') and not c.startswith('dir_')]
FOLDS=[{'year':2021,'train_end':'2019-12-24','val_start':'2020-01-03','val_end':'2020-12-24','test_start':'2021-01-05','test_end':'2021-12-31'},{'year':2022,'train_end':'2020-12-24','val_start':'2021-01-05','val_end':'2021-12-24','test_start':'2022-01-03','test_end':'2022-12-30'},{'year':2023,'train_end':'2021-12-24','val_start':'2022-01-03','val_end':'2022-12-23','test_start':'2023-01-03','test_end':'2023-12-29'},{'year':2024,'train_end':'2022-12-23','val_start':'2023-01-03','val_end':'2023-12-22','test_start':'2024-01-02','test_end':'2024-12-31'},{'year':2025,'train_end':'2023-12-22','val_start':'2024-01-02','val_end':'2024-12-24','test_start':'2025-01-02','test_end':'2025-12-31'}]
VOYAGE,IDLE,WAIT_DAYS=20.0,2500.0,1.0
def fit(f,v):
 t=f'target_{v}_1d'; ok=df[v].notna()&df[t].notna(); tr=(df.date<=f['train_end'])&ok; va=(df.date>=f['val_start'])&(df.date<=f['val_end'])&ok; te=(df.date>=f['test_start'])&(df.date<=f['test_end'])&ok
 def X(m): return np.nan_to_num(df.loc[m,FEATURES].to_numpy(),nan=0,posinf=0,neginf=0)
 def y(m): return df.loc[m,t].to_numpy()-df.loc[m,v].to_numpy()
 sc=StandardScaler().fit(X(tr)); sel=SelectKBest(f_regression,k=min(30,len(FEATURES))).fit(sc.transform(X(tr)),y(tr)); rf=RandomForestRegressor(n_estimators=100,max_depth=5,random_state=42,n_jobs=1).fit(sel.transform(sc.transform(X(tr))),y(tr))
 vp=rf.predict(sel.transform(sc.transform(X(va)))); tp=rf.predict(sel.transform(sc.transform(X(te)))); return {'vp':vp,'vb':df.loc[va,v].to_numpy(),'vt':df.loc[va,t].to_numpy(),'tp':tp,'b':df.loc[te,v].to_numpy(),'a':df.loc[te,t].to_numpy(),'d':df.loc[te,'date'].dt.strftime('%Y-%m-%d').to_numpy()}

In [ ]:
rows=[]; thresholds=[]
for f in FOLDS:
 for v in VESSELS:
  r=fit(f,v); cand=np.linspace(-300,-25,50); nets=[np.sum(r['vb']*VOYAGE-np.where(r['vp']<x,r['vt']*VOYAGE+IDLE,r['vb']*VOYAGE)) for x in cand]; tau=float(cand[int(np.argmax(nets))]); thresholds.append({'year':f['year'],'vessel':v,'train_end':f['train_end'],'val_start':f['val_start'],'val_end':f['val_end'],'test_start':f['test_start'],'test_end':f['test_end'],'threshold':tau})
  for i in range(len(r['a'])):
   b,a,p=r['b'][i],r['a'][i],r['tp'][i]; spot=b*VOYAGE; wait=a*VOYAGE+IDLE*WAIT_DAYS
   d04='WAIT' if p < -125.0 else 'NON_WAIT'; d06='WAIT' if p < tau else 'NON_WAIT'; rows.append({'date':r['d'][i],'year':f['year'],'vessel':v,'pred_delta':p,'threshold':tau,'base_rate':b,'true_rate':a,'realized_delta':a-b,'EXP04_decision':d04,'EXP06_decision':d06,'spot_cost':spot,'WAIT_cost':wait,'EXP04_contribution':spot-wait if d04=='WAIT' else 0.0,'EXP06_contribution':spot-wait if d06=='WAIT' else 0.0})
rows=pd.DataFrame(rows); thresholds=pd.DataFrame(thresholds); assert len(rows)==4804; assert int((rows.EXP04_decision=='WAIT').sum())==1375; assert int((rows.EXP06_decision=='WAIT').sum())==1509
rows['paired_delta_contribution']=rows.EXP06_contribution-rows.EXP04_contribution; observed_exp04=float(rows.EXP04_contribution.sum()); observed_exp06=float(rows.EXP06_contribution.sum()); observed_delta=float(rows.paired_delta_contribution.sum())
print({'EXP04_total_economic_value':observed_exp04,'EXP06_total_economic_value':observed_exp06,'observed_delta':observed_delta,'EXP04_WAIT_N':int((rows.EXP04_decision=='WAIT').sum()),'EXP06_WAIT_N':int((rows.EXP06_decision=='WAIT').sum())})
assert abs(observed_delta-130480.0)<1e-6

## Permutation null

The null randomizes each policy's WAIT assignment independently within each vessel/year stratum while holding fixed: the 4,804 rows, row-level economic inputs, the 20 stratum definitions, and each policy's observed WAIT count in every stratum. Thus the unequal global volumes, 1,375 and 1,509, are preserved. No model is retrained and no future target is used to select a threshold or assignment.

In [ ]:
DRAWS=10000; SEED=20260925; rng=np.random.default_rng(SEED); strata=[g.index.to_numpy() for _,g in rows.groupby(['year','vessel'],sort=True)]
n04={s:int((rows.loc[s,'EXP04_decision']=='WAIT').sum()) for s in strata}; n06={s:int((rows.loc[s,'EXP06_decision']=='WAIT').sum()) for s in strata}; values=rows.paired_delta_contribution.to_numpy(); placebo=np.empty(DRAWS)
for draw in range(DRAWS):
 total=0.0
 for s in strata:
  i04=rng.choice(s,size=n04[s],replace=False); i06=rng.choice(s,size=n06[s],replace=False); total += values[i06].sum()-values[i04].sum()
 placebo[draw]=total
ge=int(np.sum(np.abs(placebo)>=abs(observed_delta))); pvalue=(ge+1)/(DRAWS+1)
summary=pd.DataFrame([{'observed_delta':observed_delta,'placebo_mean':placebo.mean(),'placebo_std':placebo.std(ddof=1),'placebo_2.5_percentile':np.percentile(placebo,2.5),'placebo_97.5_percentile':np.percentile(placebo,97.5),'empirical_two_sided_p_value':pvalue,'placebo_abs_ge_observed_N':ge,'draws':DRAWS,'seed':SEED,'canonical_N':len(rows),'EXP04_WAIT_N':int((rows.EXP04_decision=='WAIT').sum()),'EXP06_WAIT_N':int((rows.EXP06_decision=='WAIT').sum())}]); display(summary)
print('count sanity:',len(rows),n04 and sum(n04.values()),n06 and sum(n06.values()),'identical row inputs:',rows[['date','year','vessel','base_rate','true_rate','spot_cost','WAIT_cost']].duplicated().sum()==0)

In [ ]:
import matplotlib.pyplot as plt
OUT=REPO/'outputs'/'final_policy_permutation'; OUT.mkdir(parents=True,exist_ok=True)
rows.to_csv(OUT/'observed_policy_delta.csv',index=False); pd.DataFrame({'draw':np.arange(1,DRAWS+1),'placebo_delta':placebo}).to_csv(OUT/'placebo_distribution.csv',index=False); summary.to_csv(OUT/'permutation_summary.csv',index=False); thresholds.to_csv(OUT/'threshold_audit.csv',index=False)
method='''Null: independently reassign EXP-04 and EXP-06 WAIT labels within each year x vessel stratum. Held fixed: canonical 4,804 rows, row-level spot and WAIT economics, model predictions, learned thresholds, stratum-specific WAIT counts, and unequal totals 1,375 versus 1,509. No retraining and no test-period information enters the permutation. Each draw is placebo_EXP06_value - placebo_EXP04_value. Two-sided p=(1+#|placebo|>=|observed|)/(10000+1).
'''; (OUT/'methodology.txt').write_text(method)
plt.figure(figsize=(9,5)); plt.hist(placebo,bins=60,alpha=.75); plt.axvline(observed_delta,color='red',lw=2,label=f'observed {observed_delta:,.0f}'); plt.axvline(-observed_delta,color='red',lw=1,ls='--'); plt.legend(); plt.title('Stratified paired placebo deltas'); plt.xlabel('EXP-06 minus EXP-04 economic value'); plt.ylabel('draw count'); plt.tight_layout(); plt.savefig(OUT/'placebo_distribution.png',dpi=160); plt.show()
print('outputs:',OUT); [print(p.name,hashlib.sha256(p.read_bytes()).hexdigest()) for p in sorted(OUT.iterdir()) if p.is_file()]